# 🧬 MTP ヘッドの形を safetensors ヘッダから確認

`method=mtp` の汎用指定で拾えるかの判断材料。


In [ ]:
# Dataset の中身から MTP ヘッドの形を確認（10 秒）
# `method=mtp` が汎用指定で拾えるか、draft model 指定が要るかの判断材料。
import json
import os
import struct
import sys

ROOT = ("/kaggle/input/datasets/yamahei21python/"
        "qwen38-27b-uncensored-hauhaucs-awq-mtp")
print("exists:", os.path.isdir(ROOT))
if not os.path.isdir(ROOT):
    cands = [p for p in ("/kaggle/input/qwen38-27b-uncensored-hauhaucs-awq-mtp",
                         "/kaggle/input/datasets/yamahei21python/"
                         "qwen38-27b-uncensored-hauhaucs-awq-mtp") if os.path.isdir(p)]
    print("cands:", cands)
    if cands:
        ROOT = cands[0]

cfg = json.load(open(os.path.join(ROOT, "config.json")))
print("architectures:", cfg.get("architectures"))
print("model_type   :", cfg.get("model_type"))
print("layers       :", cfg.get("num_hidden_layers"))
print("hidden       :", cfg.get("hidden_size"))
print("quantization :", cfg.get("quantization_config"))
print("max_pos      :", cfg.get("max_position_embeddings"))
print("tie_embed    :", cfg.get("tie_word_embeddings"))


def header(path):
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        return json.loads(f.read(n).decode("utf-8", "replace"))


for name in ("model-mtp.safetensors", "model.safetensors"):
    p = os.path.join(ROOT, name)
    if not os.path.isfile(p):
        print("missing:", name)
        continue
    h = header(p)
    keys = [k for k in h if k != "__metadata__"]
    print(f"\n== {name}: {len(keys)} tensors ==")
    print("metadata:", h.get("__metadata__"))
    # 層番号の分布（MTP が何層目に足されているか）
    layers = sorted({int(k.split(".")[2]) for k in keys
                     if k.startswith("model.layers.") and k.split(".")[2].isdigit()})
    print("layer ids:", layers[:5], "...", layers[-5:] if layers else None,
          "count:", len(layers))
    print("first keys:", keys[:12])
    if "mtp" in name:
        print("ALL MTP KEYS:")
        for k in keys:
            print("  ", k, h[k].get("shape"), h[k].get("dtype"))

gen = os.path.join(ROOT, "generation_config.json")
if os.path.isfile(gen):
    print("\ngeneration_config:", json.load(open(gen)))
